> Optional historical reference. Not part of the current teaching path. Check API/model compatibility before running; use the root locked course for validated examples.


In [ ]:
import os
import getpass

from dotenv import load_dotenv

# Reads OPENAI_API_KEY from a local .env file (see .env.example).
# Falls back to an interactive prompt if the variable still isn't set.
load_dotenv()

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

In [ ]:
from openai import OpenAI
from pydantic import BaseModel, Field

client = OpenAI()

class RouterTag(BaseModel):
    input_tag: str = Field(..., description="The tag with a classification of the input to process")
    

def classify_task(prompt: str) -> str:
    response = client.responses.parse(
        model="gpt-5.6-luna",
        instructions="You classify tasks into 3 different categories: 'engineering', 'marketing', 'sales'",
        input=prompt,
        text_format=RouterTag
    )
    return response.output_parsed

print(classify_task("Fix bug with landing page"))
print(classify_task("Design a new promotional campaign for our latest product"))  # marketing example
print(classify_task("Reach out to potential clients to close deals"))             # sales example

In [ ]:
def engineering_agent(task: str):
    """
    Engineering agent: Specializes in software development, bug fixes, and technical analysis.
    Tools available: codebase access, bug tracker, documentation.
    """
    response = client.responses.create(
        model="gpt-5.6-luna",
        instructions=(
            "You are an engineering expert, skilled at solving software problems, fixing bugs, "
            "and providing technical solutions. Always use precise technical language, "
            "cite documentation when relevant, and reference relevant code modules. "
            "Your outputs at this first stage are short one paragraph plans for the task."
        ),
        input=task,
        
    )
    return response.output_text

def marketing_agent(task: str):
    """
    Marketing agent: Specializes in creating campaigns, content, and brand strategies.
    Tools available: campaign planner, analytics dashboard, content calendar.
    """
    client = OpenAI()
    response = client.responses.parse(
        model="gpt-5.6-luna",
        instructions=(
            "You are a marketing expert, focused on campaign creation, brand strategy, "
            "and content ideation. Use persuasive and creative language, and connect ideas "
            "to business objectives. "
            "Your outputs at this first stage are short one paragraph plans for the task."
        ),
        input=task
    )
    return response.output_text

def sales_agent(task: str):
    """
    Sales agent: Specializes in outreach, client management, and closing deals.
    """
    client = OpenAI()
    response = client.responses.create(
        model="gpt-5.6-luna",
        instructions=(
            "You are a sales expert with skill in outreach, managing leads, "
            "and progressing conversations toward closing deals. "
            "Use persuasive, professional language and reference tools when needed. "
            "Your outputs at this first stage are short one paragraph plans for the task."
        ),
        input=task
    )
    return response.output_text


def router(prompt: str) -> str:
    tag = classify_task(prompt)
    print("Classified task into: ", tag.input_tag)
    if tag.input_tag == "engineering":
        return engineering_agent(prompt)
    elif tag.input_tag == "marketing":
        return marketing_agent(prompt)
    elif tag.input_tag == "sales":
        return sales_agent(prompt)
    else:
        return "No matching agent found"

# example for an engineering task
print(router("Fix bug with landing page"))
# print(router("Design a new promotional campaign for our latest product"))  # marketing example
# print(router("Reach out to potential clients to close deals"))             # sales example